# Train 4 góc CCCD trên Kaggle GPU
Dùng với `card_pose_kaggle.zip` được tạo từ project. Giữ **Dataset và Notebook Private** vì ảnh vẫn chứa thông tin cá nhân.
Bật GPU trong Settings → Accelerator. Bật Internet cho bước cài thư viện và thêm dataset đã upload.
Có thể Run All rồi tải file của phiên hiện tại; hoặc Save Version → Save & Run All để chạy nền và lưu kết quả.
Notebook fine-tune model đang có; GPU giúp train nhanh hơn, không sửa nhãn sai hoặc thiếu đa dạng dữ liệu.
Không chạy `uv sync` hay cài cả service: cấu hình Linux của project chọn Torch CPU.

In [ ]:
import os
import subprocess
import sys

os.environ["YOLO_CONFIG_DIR"] = "/tmp/ekyc-yolo-settings"
os.environ["YOLO_AUTOINSTALL"] = "false"
os.environ["YOLO_OFFLINE"] = "true"
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ultralytics==8.4.208",
        "pydantic-settings>=2.16.2,<3",
        "httpx",
        "PyYAML",
    ]
)

## 1. Kiểm tra GPU
Dùng một GPU (`device=0`); không cần cấu hình multi-GPU cho tập dữ liệu nhỏ này.

In [ ]:
import torch
import ultralytics

assert torch.cuda.is_available(), "Chưa có CUDA. Bật GPU trong Settings rồi restart session."
print(
    {
        "torch": torch.__version__,
        "ultralytics": ultralytics.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "cuda": torch.version.cuda,
    }
)

## 2. Tìm gói dữ liệu, kiểm tra hash, sửa đường dẫn
Hỗ trợ cả trường hợp Kaggle tự giải nén ZIP và ZIP còn nguyên trong `/kaggle/input`.
Ảnh/nhãn/code được chép vào `/tmp`, không đưa ảnh vào thư mục Output `/kaggle/working`.
Checkpoint và lịch sử train vẫn ghi vào `/kaggle/working` để lưu cùng phiên bản notebook.

In [ ]:
import hashlib
import json
import shutil
import zipfile
from pathlib import Path

import yaml

KIND = "ekyc-card-pose-kaggle-v1"
input_root = Path("/kaggle/input")
markers = []
for marker in input_root.rglob("bundle_manifest.json"):
    try:
        if json.loads(marker.read_text())["kind"] == KIND:
            markers.append(marker)
    except (ValueError, KeyError):
        pass
if len(markers) == 1:
    manifest_bytes = markers[0].read_bytes()
    manifest = json.loads(manifest_bytes)
    job_id = hashlib.sha256(manifest_bytes).hexdigest()[:12]
    ROOT = Path("/tmp/ekyc-card-pose") / job_id / "card_pose_kaggle"
    shutil.copytree(markers[0].parent, ROOT, dirs_exist_ok=True)
elif len(markers) == 0:
    archives = list(input_root.rglob("card_pose_kaggle.zip"))
    assert len(archives) == 1, "Thêm một bundle, hoặc sửa đường dẫn ZIP trong archives."
    with zipfile.ZipFile(archives[0]) as archive:
        manifest_bytes = archive.read("card_pose_kaggle/bundle_manifest.json")
        manifest = json.loads(manifest_bytes)
        assert manifest["kind"] == KIND
        job_id = hashlib.sha256(manifest_bytes).hexdigest()[:12]
        destination = Path("/tmp/ekyc-card-pose") / job_id
        for name in archive.namelist():
            resolved = (destination / name).resolve()
            assert resolved.is_relative_to(destination.resolve()), "Unsafe ZIP path"
        archive.extractall(destination)
        ROOT = destination / "card_pose_kaggle"
else:
    raise RuntimeError("Có nhiều bundle. Chỉ gắn một bundle hoặc chọn markers[0] rõ ràng.")

for name, expected in manifest["files"].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == expected, (
        f"Hash mismatch: {name}"
    )
DATA = ROOT / "dataset" / "data.yaml"
data = yaml.safe_load(DATA.read_text())
data["path"] = str(ROOT / "dataset")
DATA.write_text(yaml.safe_dump(data))
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
OUTPUT = Path("/kaggle/working")
OUTPUT.mkdir(exist_ok=True)
print(
    {
        "dataset_counts": manifest["counts"],
        "data": str(DATA),
        "label_quality": manifest["label_quality"],
    }
)

## 3. Train bằng GPU
Thông số là điểm khởi đầu để fine-tune `models/seed.pt`, không phải bộ thông số đã chứng minh cải thiện.
Giữ nguyên split theo nhóm thẻ; hiện có 37 ảnh train cùng thẻ và 16 ảnh validation.
Không bật flip vì thứ tự nhãn góc theo vị trí hình học. Có thể giảm BATCH khi hết VRAM.
Script chọn best/last bằng sai số góc và phạt dự đoán không qua hình học; bước chọn chạy CPU như service.

In [ ]:
EPOCHS = 120
BATCH = 16
IMGSZ = 640
WEIGHTS = ROOT / "models" / "seed.pt"
MODEL = OUTPUT / "card_yolov8n_pose.kaggle.pt"
command = [
    sys.executable,
    str(ROOT / "scripts" / "train_card_detector.py"),
    str(DATA),
    "--weights",
    str(WEIGHTS),
    "--device",
    "0",
    "--workers",
    "2",
    "--epochs",
    str(EPOCHS),
    "--batch",
    str(BATCH),
    "--imgsz",
    str(IMGSZ),
    "--lr0",
    "0.0001",
    "--degrees",
    "15",
    "--pose-weight",
    "12",
    "--patience",
    "30",
    "--project",
    str(OUTPUT / "card_pose_training"),
    "--output",
    str(MODEL),
]
subprocess.run(command, check=True)

## 4. Xem kết quả chọn checkpoint
Không dùng mAP hoặc việc train thành công để kết luận crop đúng.
`mean_error` gồm phạt 1.0 cho ảnh không qua hình học; `valid_mean_corner_error` chỉ tính ảnh qua.
Những số này tính trên validation đã dùng chọn model, chưa phải test độc lập.

In [ ]:
from ultralytics import YOLO

selection = json.loads(MODEL.with_suffix(".selection.json").read_text())
print(json.dumps(selection, indent=2))
verified = YOLO(str(MODEL))
assert verified.names == {0: "card"}
assert verified.model.yaml.get("kpt_shape") == [4, 3]

## 5. Đóng gói trọng số để tải về
Gói tải chỉ gồm model, metadata và bảng train; không kèm ảnh.
Nếu đã Run All xong, tải `card_detector_download.zip` từ Output của phiên hiện tại hoặc link dưới.
Save Version → Save & Run All dùng để chạy lại toàn bộ và lưu kết quả; không bắt buộc train lại chỉ để tải file.
Có thể tải trực tiếp model `.pt`. Giữ notebook Private.

In [ ]:
from IPython.display import FileLink, display

training_report = json.loads(MODEL.with_suffix(".json").read_text())
run_folder = Path(training_report["best"]).parent.parent
provenance = {
    "bundle_manifest_sha256": hashlib.sha256(manifest_bytes).hexdigest(),
    "dataset_counts": manifest["counts"],
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "model_sha256": hashlib.sha256(MODEL.read_bytes()).hexdigest(),
    "experimental": True,
}
provenance_path = OUTPUT / "kaggle_training_provenance.json"
provenance_path.write_text(json.dumps(provenance, indent=2))
download = OUTPUT / "card_detector_download.zip"
with zipfile.ZipFile(download, "w", zipfile.ZIP_DEFLATED) as archive:
    for file in [
        MODEL,
        MODEL.with_suffix(".json"),
        MODEL.with_suffix(".selection.json"),
        provenance_path,
        run_folder / "results.csv",
        run_folder / "args.yaml",
    ]:
        if file.is_file():
            archive.write(file, file.name)
print({"download": str(download), "model_sha256": provenance["model_sha256"]})
display(FileLink(str(download)))

## 6. Dùng lại trên Windows
Giải nén `card_detector_download.zip` vào `F:/ekyc-service/models/kaggle/`.
Trong PowerShell tại project:
```powershell
$env:CARD_DETECTOR_BACKEND="yolo"
$env:CARD_YOLO_MODEL_PATH="F:/ekyc-service/models/kaggle/card_yolov8n_pose.kaggle.pt"
$env:QUALITY_PROFILE="evaluation"
uv run python scripts/evaluate_card_preparer.py tests/fixtures/private/card_pose_dataset/annotations.json --model models/kaggle/card_yolov8n_pose.kaggle.pt --output tests/fixtures/private/card_pose_dataset/kaggle_eval
```
Xem crop/overlay và so với model cũ trước khi dùng cho service.
Trọng số `.pt` train trên GPU vẫn được service nạp để inference CPU.